In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
import numpy as np
from collections import Counter
from scipy import sparse
import anndata as ad
import pickle
import matplotlib.pyplot as plt
from torch import nn

In [ ]:
he_id = 'MALT'
with open(f'./DBiT_{he_id}_gene_exp_pd_spatialDE.pickle', 'rb') as handle:
    dbit_pd = pickle.load(handle)

In [ ]:
dbit_pd

In [ ]:
x = [int(item.split('x')[0][1:]) for item in dbit_pd.index]
y = [int(item.split('x')[1]) for item in dbit_pd.index]

index_update = [str(x*2-1)+'x'+str(y*2-1) for (x,y) in zip(x,y)]
dbit_pd.index = index_update

dbit_pd

In [ ]:
with open(f'./{he_id}_cellvit_embed.pickle', 'rb') as handle:
    final_embed_norm_recon_df = pickle.load(handle)
final_embed_norm_recon_df

In [ ]:
import json
with open(f'/gpfs/ycga/work/fan/yl2499/CellViT/DLBCL_FL/preprocessing_ZB/SR20-6239_02-01_6_3811/SR20-6239_02-01_6_3811/cell_detection/cells.json') as f:
    cell_data = json.load(f)

## extract one patch from cell_data
cell_centroid_0 = []
cell_centroid_1 = []
for cell in cell_data['cells']:
    cell_centroid_0.append(cell['centroid'][0])
    cell_centroid_1.append(cell['centroid'][1])

In [ ]:
selected_cells = (np.array(cell_centroid_0) <= 40000) *(np.array(cell_centroid_1) <= 25000)
cell_centroid_0 = np.array(cell_centroid_0)[selected_cells]
cell_centroid_1 = np.array(cell_centroid_1)[selected_cells]

In [ ]:
def rotate_function(x, y, deg):
    xI_scale = x
    yI_scale = y
    ## rotate DBiT ROI
    theta_deg = deg
    theta0 = (np.pi/180)*-theta_deg

    #rotation matrix
    #rotates about the origin
    L = np.array([[np.cos(theta0),-np.sin(theta0)],
                  [np.sin(theta0),np.cos(theta0)]])

    source_L = np.matmul(L , np.array([xI_scale, yI_scale]))
    xI_L = source_L[0]
    yI_L = source_L[1]

    #translation matrix
    #effectively makes the rotation about the centroid of I (i.e the means of xI and yI])
    #and also moves the centroid of I to the centroid of J
    #T = np.array([ np.mean(xI_scale)- np.cos(theta0)*np.mean(xI_scale) +np.sin(theta0)*np.mean(yI_scale) - (np.mean(xI_scale)-np.mean(xJ)),
    #              np.mean(yI_scale)- np.sin(theta0)*np.mean(xI_scale) -np.cos(theta0)*np.mean(yI_scale) - (np.mean(yI_scale)-np.mean(yJ))])

    xI_L_T = xI_L# + T[0] - 500.
    yI_L_T = yI_L# + T[1] + 200.
    
    return xI_L_T, yI_L_T

In [ ]:
cell_centroid_0_rot, cell_centroid_1_rot = rotate_function(cell_centroid_0, cell_centroid_1, 32.5)

In [ ]:
final_embed_norm_recon_df['x'] = cell_centroid_0_rot
final_embed_norm_recon_df['y'] = cell_centroid_1_rot
final_embed_norm_recon_df

In [ ]:
with open(f'./{he_id}_cellvit_embed_rotated.pickle', 'wb') as handle:
    pickle.dump(final_embed_norm_recon_df, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
adata = ad.AnnData(sparse.csc_matrix(dbit_pd.values))
adata.var_names = list(dbit_pd.columns)
adata.obs_names = np.array(dbit_pd.index)

In [ ]:
with open(f'./{he_id}_dbit_coords.pickle', 'rb') as handle:
    visium_coords = pickle.load(handle)

In [ ]:
visium_coords.index = visium_coords.Barcode
visium_coords

In [ ]:
visium_coords_sub = visium_coords.loc[adata.obs_names, :].copy()
visium_coords_sub

In [ ]:
adata.obs['array_row'] = np.array(visium_coords_sub.array_row)
adata.obs['array_col'] = np.array(visium_coords_sub.array_col)
adata.obs['x'] = np.array(visium_coords_sub.pxl_col_in_fullres)
adata.obs['y'] = np.array(visium_coords_sub.pxl_row_in_fullres)
adata.obsm['spatial'] = np.array([np.array(visium_coords_sub.pxl_col_in_fullres), np.array(visium_coords_sub.pxl_row_in_fullres)]).T
adata.layers['counts'] = adata.X.copy()

In [ ]:
adata

In [ ]:
gene_list = list(dbit_pd.columns)

In [ ]:
import sys
import os

# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("../SuperFocus")

# Add the directory to sys.path
sys.path.append(script_dir)

In [ ]:
from importlib import reload
import src.model_training.model_training_orig
import src.model_training.processing_image_features
reload(src.model_training.model_training_orig)
reload(src.model_training.processing_image_features)
from src.model_training.model_training_orig import training_orig

In [ ]:
sample_id = 'MALT'
he_id = 'SR20-6239_02-01_6_3811'

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training_orig(adata=adata, pt_model_dir=f'../CellViT/DLBCL_FL/preprocessing_ZB/{he_id}/{he_id}/cell_detection', radius=75./2, feature_list=gene_list, 
              save_model_dir=f'./logs_{sample_id}_reproduce', hidden_dims=[8192, 4096, 4096, 8192], activations=None, residual=False, norm=False, out_act=None, 
              include_tanhshrink=False, sample_id=sample_id, training_mode='Full_3', selected_model='lite_shrink', kfold=5, cell_embed_dir=f'./{sample_id}_cellvit_embed_rotated.pickle', sort_way='col_row')

In [ ]:
import src.post_process
from importlib import reload
reload(src.post_process)
from src.post_process import post_training_processing

In [ ]:
adata_pred = post_training_processing.post_procesing(model_dir=f'./logs_{sample_id}_reproduce/checkpoints/model.best.pth', hidden_dims=[8192, 4096, 4096, 8192], 
                                               activations=None, out_act=None, residual=False, norm=False, selected_model='lite_shrink', include_tanhshrink=False, 
                                               data_dir=f'./{sample_id}_training_data_level1.pickle', feature_list=gene_list, datatype='RNA', regress_out=False, resolutions=[0.6, 0.8],
                                               refine_factor=1, sample_id=sample_id, radius=75./2)

In [ ]:
sc.pl.spatial(adata_pred, img_key=None, color='leiden_res_0.8')

In [ ]:
import src.model_training.model_training_refined
reload(src.model_training.model_training_refined)
from src.model_training.model_training_refined import training

In [ ]:
with open(f'./{sample_id}_gene_exp_pd.pickle', 'rb') as f:
    dbit_pd = pickle.load(f)
f.close()
dbit_pd

In [ ]:
from catalyst.utils import set_global_seed
set_global_seed(123)
training(adata=adata, pt_model_dir=f'../CellViT/DLBCL_FL/preprocessing_ZB/{he_id}/{he_id}/cell_detection', radius=75./2, refine_factor=2, 
         feature_pd_dir=f'./{sample_id}_gene_exp_pd.pickle', 
         trained_model_dir=f'./logs_{sample_id}_reproduce/checkpoints/model.best.pth',
         save_model_dir=f'./logs_{sample_id}_reproduce_2x', hidden_dims=[8192, 4096, 4096, 8192], activations=None, residual=False, norm=False, out_act=None, 
         include_tanhshrink=False, sample_id=sample_id, training_mode='Full_2', selected_model='lite_shrink', kfold=5, cell_embed_dir=f'./{sample_id}_cellvit_embed_rotated.pickle', 
         lr=1e-7, step_size=25, gamma=0.5, num_epochs=50, sort_way='col_row', random_state=42, datatype='dbit')